
# BTS Flight Data — Initial Data Audit

## 1. Objective

This notebook explores the raw BTS Reporting Carrier On-Time Performance dataset and validates its suitability for the Flight Delay ML Platform.

The platform supports two binary classification tasks:

- **Departure Delay:** `DepDel15`
- **Arrival Delay:** `ArrDel15`

Both predictions are made 24 hours before scheduled gate departure.

## 2. Pilot Dataset

- Source: Bureau of Transportation Statistics (BTS)
- Table: Reporting Carrier On-Time Performance
- Period: January 2023
- Format: ZIP containing CSV
- Storage location: `data/raw/`

Official source:
https://transtats.bts.gov/PREZIP/

## 3. Data Engineering Principles

1. Preserve the original source files.
2. Keep raw datasets out of GitHub.
3. Validate the schema before transformation.
4. Separate raw data from processed datasets.
5. Record data quality issues before modeling.
6. Prevent current-flight outcome information from entering model features.

## 4. Initial Workflow

Download → Inspect ZIP → Verify CSV Schema → Data Audit → ETL → Parquet

The pilot uses one month of data before expanding to the full 2023–2025 dataset.


In [2]:
from pathlib import Path
from urllib.request import Request, urlopen
import shutil
import zipfile

# Find repository root
current = Path.cwd().resolve()

ROOT = next(
    (
        path for path in [current, *current.parents]
        if (path / "pyproject.toml").exists()
    ),
    None,
)

assert ROOT is not None, "Project root not found."

RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2023
MONTH = 1

filename = (
    "On_Time_Reporting_Carrier_On_Time_Performance_"
    f"1987_present_{YEAR}_{MONTH}.zip"
)

url = f"https://transtats.bts.gov/PREZIP/{filename}"
zip_path = RAW_DIR / filename

# Download only when the local ZIP is not already valid.
if not zip_path.exists() or not zipfile.is_zipfile(zip_path):
    print("Downloading:", url)

    request = Request(
        url,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    temp_path = zip_path.with_suffix(".zip.part")

    try:
        with urlopen(request, timeout=90) as response:
            with temp_path.open("wb") as output:
                shutil.copyfileobj(response, output)

        if not zipfile.is_zipfile(temp_path):
            raise ValueError("Downloaded file is not a valid ZIP.")

        temp_path.replace(zip_path)

    finally:
        temp_path.unlink(missing_ok=True)

else:
    print("Valid ZIP already exists. Skipping download.")

print("\nSaved to:", zip_path)
print("File size (MB):", round(zip_path.stat().st_size / 1024**2, 2))

Valid ZIP already exists. Skipping download.

Saved to: /Users/dingyuyao/Documents/Project/flight-delay-ml-platform/data/raw/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_1.zip
File size (MB): 25.81


In [3]:
import csv
import io
import zipfile

with zipfile.ZipFile(zip_path) as zf:
    csv_files = [
        name for name in zf.namelist()
        if name.lower().endswith(".csv")
    ]

    assert csv_files, "No CSV file found in ZIP."

    csv_name = max(
        csv_files,
        key=lambda name: zf.getinfo(name).file_size
    )

    print("CSV file:", csv_name)

    with zf.open(csv_name) as binary_file:
        with io.TextIOWrapper(
            binary_file,
            encoding="utf-8-sig",
            newline=""
        ) as text_file:

            reader = csv.DictReader(text_file)
            columns = reader.fieldnames

            print("\nTotal columns:", len(columns))
            print("\nColumn names:")
            print(columns)

            print("\nFirst 3 records:")
            for _, row in zip(range(3), reader):
                print({
                    key: row.get(key)
                    for key in [
                        "FlightDate",
                        "Reporting_Airline",
                        "Origin",
                        "Dest",
                        "DepDel15",
                        "ArrDel15",
                    ]
                })

# Check required fields
required_columns = {
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "CRSArrTime",
    "DepDel15",
    "ArrDel15",
    "Cancelled",
    "Diverted",
}

missing = required_columns - set(columns)

assert not missing, f"Missing columns: {missing}"

print("\nInitial schema check passed!")

CSV file: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2023_1.csv

Total columns: 110

Column names:
['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number', 'Flight_Number_Reporting_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups', 'ArrTimeBlk', 'Cancelled', 'CancellationCode', 'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime', 'Flights', 'Distance', 'DistanceGrou


## 3. Initial Data Quality Audit

### Objective

Perform an initial quality assessment of the January 2023 BTS dataset before preprocessing or feature engineering.

### Audit Scope

1. Dataset dimensions and date range
2. Data types and missing values
3. Cancelled and diverted flights
4. Departure and arrival delay label distributions
5. Eligible flights for initial modeling

### Important Considerations

- The original dataset contains flights from all reporting airports.
- The modeling scope will later be restricted to six origin airports.
- `DepDel15` and `ArrDel15` must be converted to numeric labels.
- Missing labels may be associated with cancelled or diverted flights.
- Raw records should not be modified during the audit.

The audit results will guide the preprocessing and data validation pipeline.


In [5]:
import pandas as pd
import zipfile

# Fields needed for the initial audit.
AUDIT_COLUMNS = [
    "FlightDate",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "CRSArrTime",
    "CRSElapsedTime",
    "Distance",
    "DepDelay",
    "ArrDelay",
    "DepDel15",
    "ArrDel15",
    "Cancelled",
    "Diverted",
]

# Read only selected columns directly from the ZIP.
# zip_path was defined in the previous download cell.
with zipfile.ZipFile(zip_path) as zf:
    csv_name = next(
        name for name in zf.namelist()
        if name.lower().endswith(".csv")
    )

    with zf.open(csv_name) as file:
        df = pd.read_csv(
            file,
            usecols=AUDIT_COLUMNS,
            dtype={"Flight_Number_Reporting_Airline": "string"},
            low_memory=False,
        )

# Convert dates and numeric values.
df["FlightDate"] = pd.to_datetime(
    df["FlightDate"], errors="coerce"
)

numeric_columns = [
    "CRSDepTime",
    "CRSArrTime",
    "CRSElapsedTime",
    "Distance",
    "DepDelay",
    "ArrDelay",
    "DepDel15",
    "ArrDel15",
    "Cancelled",
    "Diverted",
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Dataset Shape:", df.shape)
print("Number of Rows:", len(df))
print("Number of Columns:", len(df.columns))

print("\nDate Range:")
print(df["FlightDate"].min(), "to", df["FlightDate"].max())

print("\nData Types:")
print(df.dtypes)

print("\nFirst 5 Records:")
display(df.head())

Dataset Shape: (538837, 14)
Number of Rows: 538837
Number of Columns: 14

Date Range:
2023-01-01 00:00:00 to 2023-01-31 00:00:00

Data Types:
FlightDate           datetime64[us]
Reporting_Airline               str
Origin                          str
Dest                            str
CRSDepTime                    int64
DepDelay                    float64
DepDel15                    float64
CRSArrTime                    int64
ArrDelay                    float64
ArrDel15                    float64
Cancelled                   float64
Diverted                    float64
CRSElapsedTime              float64
Distance                    float64
dtype: object

First 5 Records:


,FlightDate,Reporting_Airline,Origin,Dest,CRSDepTime,DepDelay,DepDel15,CRSArrTime,ArrDelay,ArrDel15,Cancelled,Diverted,CRSElapsedTime,Distance
0,2023-01-02,9E,BDL,LGA,800,-3.0,0.0,905,-12.0,0.0,0.0,0.0,65.0,101.0
1,2023-01-03,9E,BDL,LGA,800,-5.0,0.0,905,-8.0,0.0,0.0,0.0,65.0,101.0
2,2023-01-04,9E,BDL,LGA,800,-5.0,0.0,905,-21.0,0.0,0.0,0.0,65.0,101.0
3,2023-01-05,9E,BDL,LGA,800,-6.0,0.0,905,-17.0,0.0,0.0,0.0,65.0,101.0
4,2023-01-06,9E,BDL,LGA,800,-1.0,0.0,905,-16.0,0.0,0.0,0.0,65.0,101.0


In [6]:
# 1. Missing values
missing_report = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Pct": (df.isna().mean() * 100).round(2),
}).sort_values("Missing_Pct", ascending=False)

print("=== Missing Value Report ===")
display(missing_report)

# 2. Cancelled and diverted flights
print("\n=== Flight Status ===")

for col in ["Cancelled", "Diverted"]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

# 3. Restrict the modeling cohort to six origin airports.
ORIGIN_AIRPORTS = [
    "ATL", "DFW", "DEN", "ORD", "LAX", "SFO"
]

airport_mask = df["Origin"].isin(ORIGIN_AIRPORTS)

# Only completed flights with valid labels.
eligible_mask = (
    airport_mask
    & df["Cancelled"].eq(0)
    & df["Diverted"].eq(0)
    & df["DepDel15"].isin([0, 1])
    & df["ArrDel15"].isin([0, 1])
)

modeling_df = df.loc[eligible_mask].copy()

print("\n=== Modeling Cohort ===")
print("Total raw records:", len(df))
print("Six-airport records:", int(airport_mask.sum()))
print("Eligible completed flights:", len(modeling_df))

# 4. Target distributions
for target in ["DepDel15", "ArrDel15"]:
    print(f"\n=== {target} ===")

    counts = modeling_df[target].value_counts(
        dropna=False
    ).sort_index()

    print(counts)

    if len(modeling_df) > 0:
        delay_rate = modeling_df[target].mean() * 100
        print(f"Positive class rate: {delay_rate:.2f}%")
    else:
        print("No eligible records.")

=== Missing Value Report ===


,Missing_Count,Missing_Pct
ArrDelay,11640,2.16
ArrDel15,11640,2.16
DepDelay,9982,1.85
DepDel15,9982,1.85
FlightDate,0,0.00
Reporting_Airline,0,0.00
Origin,0,0.00
Dest,0,0.00
CRSDepTime,0,0.00
CRSArrTime,0,0.00



=== Flight Status ===

Cancelled:
Cancelled
0.0    528542
1.0     10295
Name: count, dtype: int64

Diverted:
Diverted
0.0    537492
1.0      1345
Name: count, dtype: int64

=== Modeling Cohort ===
Total raw records: 538837
Six-airport records: 115330
Eligible completed flights: 112548

=== DepDel15 ===
DepDel15
0.0    86271
1.0    26277
Name: count, dtype: int64
Positive class rate: 23.35%

=== ArrDel15 ===
ArrDel15
0.0    84482
1.0    28066
Name: count, dtype: int64
Positive class rate: 24.94%



## 4. Label Consistency and Missingness Audit

### Objective

Investigate missing target labels and verify that the binary delay indicators are consistent with the corresponding delay duration fields.

### Audit Questions

1. Are missing departure and arrival labels associated with cancelled or diverted flights?
2. Are `DepDel15` and `ArrDel15` consistent with the 15-minute delay definition?
3. Are there any unexpected label values?
4. Are there potential duplicate flight records?

### Validation Rules

For records with valid outcomes:

- `DepDel15 = 1` when `DepDelay >= 15`.
- `DepDel15 = 0` when `DepDelay < 15`.
- `ArrDel15 = 1` when `ArrDelay >= 15`.
- `ArrDel15 = 0` when `ArrDelay < 15`.

Missing values must not be automatically converted to zero.

Potential duplicate records will be investigated using flight identifiers rather than treating identical feature values as sufficient evidence of duplication.


In [8]:
# Compare missing labels across flight outcomes.

for target in ["DepDel15", "ArrDel15"]:
    print(f"\n=== Missingness Audit: {target} ===")

    audit = df.assign(
        Label_Missing=df[target].isna()
    )

    status_report = (
        audit.groupby(
            ["Cancelled", "Diverted"],
            dropna=False
        )["Label_Missing"]
        .agg(
            Total_Flights="size",
            Missing_Labels="sum",
            Missing_Rate="mean"
        )
        .reset_index()
    )

    status_report["Missing_Rate"] = (
        status_report["Missing_Rate"] * 100
    ).round(2)

    display(status_report)

    # Missing labels among normal completed flights.
    normal_mask = (
        df["Cancelled"].eq(0)
        & df["Diverted"].eq(0)
    )

    unexpected_missing = (
        normal_mask & df[target].isna()
    ).sum()

    print(
        "Missing labels in completed flights:",
        unexpected_missing
    )


=== Missingness Audit: DepDel15 ===


,Cancelled,Diverted,Total_Flights,Missing_Labels,Missing_Rate
0,0.0,0.0,527197,0,0.00
1,0.0,1.0,1345,0,0.00
2,1.0,0.0,10295,9982,96.96


Missing labels in completed flights: 0

=== Missingness Audit: ArrDel15 ===


,Cancelled,Diverted,Total_Flights,Missing_Labels,Missing_Rate
0,0.0,0.0,527197,0,0.0
1,0.0,1.0,1345,1345,100.0
2,1.0,0.0,10295,10295,100.0


Missing labels in completed flights: 0


In [9]:
# Verify binary labels against delay minutes.

label_pairs = [
    ("DepDelay", "DepDel15"),
    ("ArrDelay", "ArrDel15"),
]

for delay_col, label_col in label_pairs:

    print(f"\n=== Validating {label_col} ===")

    valid = df[
        df[delay_col].notna()
        & df[label_col].notna()
    ].copy()

    # Expected labels based on the 15-minute rule.
    expected = (valid[delay_col] >= 15).astype(int)

    actual = valid[label_col].astype(int)

    mismatch = expected.ne(actual)

    invalid_label = ~df[label_col].dropna().isin([0, 1])

    print("Valid records:", len(valid))
    print("Label mismatches:", int(mismatch.sum()))
    print("Invalid binary labels:", int(invalid_label.sum()))

    if mismatch.any():
        print("\nExample mismatched records:")
        display(
            valid.loc[
                mismatch,
                [
                    "FlightDate",
                    "Origin",
                    "Dest",
                    delay_col,
                    label_col,
                ]
            ].head(10)
        )


=== Validating DepDel15 ===
Valid records: 528855
Label mismatches: 0
Invalid binary labels: 0

=== Validating ArrDel15 ===
Valid records: 527197
Label mismatches: 0
Invalid binary labels: 0


In [10]:
# Include flight number for a more meaningful duplicate check.

flight_key_columns = [
    "FlightDate",
    "Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
]

with zipfile.ZipFile(zip_path) as zf:
    with zf.open(csv_name) as file:
        key_df = pd.read_csv(
            file,
            usecols=flight_key_columns,
            dtype={"Flight_Number_Reporting_Airline": "string"},
            low_memory=False,
        )

# Count potential duplicate business keys.
duplicate_mask = key_df.duplicated(
    subset=flight_key_columns,
    keep=False,
)

print("Total records:", len(key_df))
print(
    "Records with repeated candidate keys:",
    int(duplicate_mask.sum())
)

print("\nExample repeated keys:")

display(
    key_df.loc[duplicate_mask]
    .sort_values(flight_key_columns)
    .head(10)
)

Total records: 538837
Records with repeated candidate keys: 0

Example repeated keys:


,FlightDate,Reporting_Airline,Flight_Number_Reporting_Airline,Origin,Dest,CRSDepTime



## 5. Data Quality Findings

### 5.1 Missing Labels

The January 2023 dataset contains 538,837 flight records.

Flight outcomes:

- Completed flights: 527,197
- Cancelled flights: 10,295
- Diverted flights: 1,345

All completed flights have valid departure and arrival delay labels.

Departure delay labels are missing for 9,982 cancelled flights, while 313 cancelled flights retain departure delay labels.

Arrival delay labels are missing for all cancelled and diverted flights.

### 5.2 Label Consistency

Both binary delay indicators were validated against the corresponding delay duration fields.

Results:

- Departure label mismatches: 0
- Arrival label mismatches: 0
- Invalid binary label values: 0

### 5.3 Duplicate Audit

No repeated candidate flight keys were found.

The candidate key consists of:

- FlightDate
- Reporting_Airline
- Flight_Number_Reporting_Airline
- Origin
- Dest
- CRSDepTime

This key will be revalidated when additional months are ingested.

### 5.4 Initial Modeling Population

For the January 2023 pilot:

- Six-airport flight records: 115,330
- Eligible completed flights: 112,548

The initial modeling cohort excludes cancelled and diverted flights and requires both delay labels to be valid.

### 5.5 Data Cleaning Decisions

1. Preserve original raw BTS records.
2. Restrict modeling data to the selected origin airports.
3. Exclude cancelled and diverted flights from the initial joint modeling cohort.
4. Never replace missing target labels with zero.
5. Validate binary labels and candidate flight keys.
6. Audit missing scheduled features before removing records.
7. Keep flight outcome fields separate from predictive features.

### 5.6 Remaining Limitations

The current audit covers January 2023 only.

The initial cohort is conditional on completed flights, so the models cannot yet provide unconditional delay risk estimates for every booked flight.

BTS retrospective schedules also require further point-in-time availability analysis before production use.


In [12]:
# Step 2.3.1 - Validate initial cleaning rules

ORIGIN_AIRPORTS = {
    "ATL", "DFW", "DEN", "ORD", "LAX", "SFO"
}

# 1. Define the initial cohort.
in_scope = df["Origin"].isin(ORIGIN_AIRPORTS)

completed = (
    df["Cancelled"].eq(0)
    & df["Diverted"].eq(0)
)

valid_labels = (
    df["DepDel15"].isin([0, 1])
    & df["ArrDel15"].isin([0, 1])
)

eligible = in_scope & completed & valid_labels

# 2. Summarize sample selection.
print("=== Sample Selection ===")

print("Raw records:", len(df))
print("Six-airport records:", int(in_scope.sum()))
print("Completed six-airport flights:",
      int((in_scope & completed).sum()))
print("Eligible modeling records:",
      int(eligible.sum()))

print("\nExcluded from initial cohort:",
      int(in_scope.sum() - eligible.sum()))

# 3. Check missing predictor candidates.
schedule_features = [
    "FlightDate",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "CRSArrTime",
    "CRSElapsedTime",
    "Distance",
]

print("\n=== Missing Scheduled Features ===")

missing = df.loc[eligible, schedule_features].isna().sum()
display(missing.to_frame("Missing_Count"))

# 4. Validate target values.
assert df.loc[eligible, "DepDel15"].isin([0, 1]).all()
assert df.loc[eligible, "ArrDel15"].isin([0, 1]).all()

print("\nInitial cleaning rules validated!")

=== Sample Selection ===
Raw records: 538837
Six-airport records: 115330
Completed six-airport flights: 112548
Eligible modeling records: 112548

Excluded from initial cohort: 2782

=== Missing Scheduled Features ===


,Missing_Count
FlightDate,0
Reporting_Airline,0
Origin,0
Dest,0
CRSDepTime,0
CRSArrTime,0
CRSElapsedTime,0
Distance,0



Initial cleaning rules validated!



## 6. Reusable Data Validation

### Objective

Convert the exploratory data quality checks into a reusable Python validation module.

**Implementation:** `src/flight_delay/data/validate.py`

### Validation Rules

1. Verify that all required BTS fields exist.
2. Check cancellation and diversion status values.
3. Validate binary delay labels.
4. Verify consistency between delay minutes and binary labels.
5. Require valid labels for completed flights.
6. Check baseline feature completeness in the eligible modeling population.

### Engineering Design

The validation function does not modify input data.

Instead, it raises explicit errors when validation rules fail and returns summary counts when the checks pass.

This fail-fast approach helps prevent invalid data from entering downstream ETL and model training pipelines.

### Pilot Reference Results

- Raw records: 538,837
- Six-airport records: 115,330
- Eligible completed flights: 112,548

These results were obtained during the January 2023 exploratory audit and will be compared with the reusable validation module.


In [14]:
from flight_delay.data.validate import validate_bts_data

# Reuse the DataFrame loaded during the initial audit.
validation_results = validate_bts_data(df)

print("=== BTS Data Validation Results ===")

for key, value in validation_results.items():
    print(f"{key}: {value:,}")

print("\nAll validation checks passed!")

=== BTS Data Validation Results ===
raw_records: 538,837
six_airport_records: 115,330
eligible_records: 112,548

All validation checks passed!



## 7. Automated Data Validation Tests

### Objective

Verify that the reusable BTS validation module correctly accepts valid data and rejects invalid records.

### Implementation

- Validation module: `src/flight_delay/data/validate.py`
- Unit tests: `tests/test_data_validation.py`
- Testing framework: pytest

### Test Coverage

The tests cover:

1. Valid BTS data acceptance
2. Missing required columns
3. Invalid binary labels
4. Inconsistent delay labels
5. Missing labels in completed flights
6. Missing baseline features
7. Invalid flight status indicators
8. Inconsistent missingness between labels and delay durations

### Testing Strategy

Synthetic test data is used to verify expected behavior without loading the entire BTS dataset.

Both positive and negative test cases are included.

### Limitations

These tests verify the current validation rules but do not yet cover candidate-key uniqueness, cross-month consistency, historical feature availability, or end-to-end pipeline behavior.

Additional tests will be implemented as the ETL pipeline evolves.



### Testing Strategy: Unit vs Integration Tests

The project uses multiple levels of validation.

**Unit Tests**

Small synthetic datasets are used to verify validation logic, including error handling, boundary cases, and invalid inputs.

**Integration Tests**

The January 2023 BTS dataset is used to verify that the validation module works with actual flight records.

Reference results:

- Raw records: 538,837
- Six-airport records: 115,330
- Eligible records: 112,548

**Data Quality Checks**

The production ETL pipeline will perform schema, label, missingness, and duplicate checks for each downloaded month.

Integration tests requiring local raw data are separated from fast unit tests because large datasets are not committed to Git.



## 8. Data Preprocessing Pipeline

### Objective

Implement a reusable preprocessing function to create the initial modeling cohort from validated BTS records.

### Implementation

`src/flight_delay/data/preprocess.py`

### Preprocessing Rules

1. Validate the input dataset before transformation.
2. Retain flights departing from ATL, DFW, DEN, ORD, LAX, or SFO.
3. Exclude cancelled and diverted flights.
4. Require valid `DepDel15` and `ArrDel15` labels.
5. Convert `FlightDate` to datetime.
6. Convert binary target labels to integer types.
7. Preserve the original input DataFrame.

### Engineering Considerations

The preprocessing function returns a new DataFrame and does not modify raw data.

The cleaned dataset retains flight outcome columns for auditing and evaluation. These fields must be excluded from the model feature matrix.

The initial modeling cohort is conditioned on completed flights and does not represent unconditional delay risk across all scheduled flights.

### Expected Pilot Results

- Raw flight records: 538,837
- Six-airport records: 115,330
- Eligible completed flights: 112,548

The preprocessing output should match the previously validated cohort count.



## 9. Preprocessing Unit Tests

### Objective

Verify that the preprocessing pipeline correctly transforms validated BTS data into the initial modeling cohort.

### Implementation

- Module: `src/flight_delay/data/preprocess.py`
- Tests: `tests/test_preprocess.py`
- Framework: pytest

### Test Cases

1. **Cohort Filtering:** Retain completed flights from the selected origin airports.
2. **Data Type Validation:** Convert dates and binary labels to the expected types.
3. **Input Preservation:** Ensure preprocessing does not modify the original DataFrame.
4. **Idempotency:** Verify repeated preprocessing produces identical results.
5. **Invalid Input Handling:** Reject inconsistent delay labels before transformation.

### Testing Strategy

Synthetic flight records are used to test normal, cancelled, diverted, and out-of-scope flights.

These unit tests focus on transformation correctness and error handling.

Real BTS records are validated separately through notebook-based checks and integration testing.

### Limitations

The current tests do not cover multi-month ingestion, Parquet persistence, distributed processing, or point-in-time feature availability.


### Results

- Preprocessing unit tests: 5/5 passed
- Ruff: All checks passed


## 10. Parquet Storage and Verification

### Objective

Persist the validated and preprocessed BTS flight records in Parquet format for downstream feature engineering and model training.

### Implementation

`src/flight_delay/data/storage.py`

### Storage Strategy

- Raw BTS ZIP files are preserved without modification.
- Cleaned records are stored in `data/processed/`.
- Parquet is used for efficient columnar storage.
- Snappy compression is applied.
- The DataFrame index is excluded from storage.
- Temporary-file writing and atomic replacement reduce the risk of partially written output files.

### Data Verification

After writing, the Parquet file will be loaded again to verify:

1. Row and column counts.
2. Column names and ordering.
3. Data types.
4. Departure and arrival target distributions.
5. Equality of the original cleaned DataFrame and the reloaded DataFrame.

### Data Leakage Considerations

Processed data retains historical outcome columns for auditing and model evaluation.

These outcome fields are not approved predictive features and must be excluded when constructing the model feature matrix.

### Version Control

Generated Parquet files are stored locally and excluded from GitHub through `.gitignore`.


In [23]:
from flight_delay.data.preprocess import prepare_modeling_cohort

# df is the raw January 2023 DataFrame
cleaned_df = prepare_modeling_cohort(df)

print("Cleaned shape:", cleaned_df.shape)
print("Cleaned data ready for Parquet storage!")

Cleaned shape: (112548, 14)
Cleaned data ready for Parquet storage!


In [25]:
from pathlib import Path

import pandas as pd
from pandas.testing import assert_frame_equal

from flight_delay.data.storage import (
    load_parquet,
    save_parquet,
)

# Locate the project root.
current = Path.cwd().resolve()

ROOT = next(
    (
        path
        for path in [current, *current.parents]
        if (path / "pyproject.toml").exists()
    ),
    None,
)

assert ROOT is not None, "Project root not found."

# cleaned_df was created by prepare_modeling_cohort().
assert "cleaned_df" in globals(), (
    "Run the preprocessing cell first."
)

# Save the January 2023 modeling cohort.
output_path = (
    ROOT / "data" / "processed"
    / "bts_2023_01_completed.parquet"
)

saved_path = save_parquet(cleaned_df, output_path)

print("Saved:", saved_path)
print(
    "File size (MB):",
    round(saved_path.stat().st_size / 1024**2, 2),
)

# Reload the Parquet file.
reloaded_df = load_parquet(saved_path)

print("\n=== Data Verification ===")

print("Original shape:", cleaned_df.shape)
print("Reloaded shape:", reloaded_df.shape)

# Verify full DataFrame equality.
assert_frame_equal(
    cleaned_df,
    reloaded_df,
    check_dtype=True,
)

# Verify expected pilot record count.
assert len(reloaded_df) == 112548

# Verify targets.
for target in ["DepDel15", "ArrDel15"]:
    print(f"\n{target} distribution:")
    print(reloaded_df[target].value_counts().sort_index())

print("\nParquet round-trip verification passed!")

Saved: /Users/dingyuyao/Documents/Project/flight-delay-ml-platform/data/processed/bts_2023_01_completed.parquet
File size (MB): 0.76

=== Data Verification ===
Original shape: (112548, 14)
Reloaded shape: (112548, 14)

DepDel15 distribution:
DepDel15
0    86271
1    26277
Name: count, dtype: int64

ArrDel15 distribution:
ArrDel15
0    84482
1    28066
Name: count, dtype: int64

Parquet round-trip verification passed!



## 11. Parquet Storage Testing

### Objective

Verify the correctness and reliability of the reusable Parquet storage module.

### Implementation

- Module: `src/flight_delay/data/storage.py`
- Tests: `tests/test_storage.py`
- Framework: pytest

### Test Cases

1. **Round-trip Integrity:** Verify that saving and loading preserves DataFrame content and data types.
2. **Directory Creation:** Verify that missing output directories are created.
3. **Idempotent Replacement:** Verify that repeated saves do not append duplicate records.
4. **Invalid Output Format:** Reject non-Parquet file extensions.
5. **Missing File Handling:** Raise an explicit error when the input file does not exist.

### Testing Strategy

Storage tests use small synthetic DataFrames and pytest temporary directories.

This isolates file-system tests from the real BTS datasets.

### Pilot Dataset Verification

The January 2023 cleaned dataset was successfully saved and reloaded.

- Records: 112,548
- Columns: 14
- Compressed file size: approximately 0.76 MB
- DataFrame round-trip equality: Passed

### Limitations

The current storage implementation supports atomic replacement of individual local files.

It does not yet provide concurrent-write coordination, dataset versioning, or distributed storage.



## 12. End-to-End Monthly ETL Pipeline

### Objective

Integrate the reusable data ingestion, validation, preprocessing, and storage components into a single executable monthly ETL pipeline.

### Implementation

`src/flight_delay/data/pipeline.py`

### Pipeline Workflow

1. Locate the monthly BTS ZIP file.
2. Read required columns from the raw CSV.
3. Verify that all flight dates belong to the requested month.
4. Validate source data quality.
5. Prepare the completed-flight modeling cohort.
6. Save the cleaned data to Parquet.
7. Reload and verify the persisted dataset.
8. Return a structured execution summary.

### Command-Line Execution

From the project root:

```bash
python -m flight_delay.data.pipeline --year 2023 --month 1
```

### Engineering Decisions

**Parameterization**

The pipeline accepts year, month, input directory, and output directory as arguments.

**Modularity**

The pipeline reuses existing validation, preprocessing, and storage functions instead of duplicating their logic.

**Fail-Fast Validation**

Invalid source data stops the pipeline before the processed output is replaced.

**Reproducibility**

The same input data and transformation rules should produce equivalent processed data.

**Logging**

The pipeline records key execution stages using Python's logging module.

### Scope and Limitations

This initial implementation processes one local BTS ZIP file at a time.

Automated downloads, multi-month orchestration, distributed processing, and persistent run metadata are planned for later phases.



## 13. Monthly ETL Pipeline Integration Testing

### Objective

Verify that the monthly ETL pipeline correctly processes BTS-style ZIP files and rejects invalid inputs.

### Implementation

- Pipeline: `src/flight_delay/data/pipeline.py`
- Tests: `tests/integration/test_pipeline.py`
- Framework: pytest

### Test Scenarios

1. Successful end-to-end execution
2. Repeatable processing without duplicate appends
3. Invalid month parameters
4. Missing source ZIP files
5. Flight dates outside the requested month
6. Inconsistent delay labels

### Testing Strategy

Synthetic BTS-style ZIP files are generated in temporary directories.

Tests invoke the real pipeline, including CSV loading, validation, preprocessing, Parquet writing, and output verification.

The January 2023 real BTS dataset is tested separately.

### Engineering Value

Integration tests help detect failures that individual module-level tests might miss.

They also establish a regression testing foundation for future multi-month ingestion.

### Remaining Limitations

- Automated monthly downloads are not yet implemented.
- Pipeline tests do not cover every possible malformed CSV or ZIP.
- Cross-month duplicate detection is not implemented.
- Historical feature availability and data leakage checks require further development.


## 14. Flight Identification and Duplicate Audit

### Objective

Preserve flight identification information and audit potential duplicate records in the processed dataset.

### Candidate Flight Key

- FlightDate
- Reporting_Airline
- Flight_Number_Reporting_Airline
- Origin
- Dest
- CRSDepTime

### Validation Strategy

The validation module reports:

1. Records with incomplete candidate keys.
2. Records sharing the same complete candidate key.

Candidate keys are used for data quality auditing and are not assumed to be guaranteed unique identifiers.

Potential duplicates are investigated rather than automatically deleted.

### Feature Engineering Consideration

Flight numbers are preserved for record tracing and auditing but are excluded from the initial model feature allowlist.

### Limitations

Candidate-key uniqueness must be re-evaluated as additional months are ingested.
